# Part 2 - Evaluating and Deploying our Agent Systems

We are going to pacakge our tools together using Langchain (in the agent.py file), and use it as a first agent version to run our evaluation!

## Agent Evaluation with MLFlow 3
Now that we've created an agent, we need to measure its performance, and find a way to compare it with previous versions.

Databricks makes it very easy with MLFlow 3. You can automatically:

- Trace all your agent input/output
- Capture end user feedback
- Evaluate your agent against custom or synthetic evaluation dataset
- Build labeled dataset with your business expert
- Compare each evaluation against the previous one
- Deploy and track your evaluations once deployed in production 

<img src="https://github.com/databricks-demos/dbdemos-resources/blob/main/images/product/ai-agent/mlflow-evaluate-0.png?raw=true?raw=true" width="800px">

### Our agent is composed of:

- [**agent.py**]($./agent.py): in this file, we used Langchain to prepare an agent ready to be used.
- [**agent_config.yaml**]($./agent_config.yaml): this file contains our agent configuration, including the system prompt and the LLM endpoint that we'll use

Let's get started and try our Langchain agent in this notebook!


<!-- Collect usage data (view). Remove it to disable collection or disable tracker during installation. View README for more details.  -->
<img width="1px" src="https://ppxrzfxige.execute-api.us-west-2.amazonaws.com/v1/analytics?category=data-science&org_id=2162748966026566&notebook=%2F02-agent-eval%2F02.1_agent_evaluation&demo_name=ai-agent&event=VIEW&path=%2F_dbdemos%2Fdata-science%2Fai-agent%2F02-agent-eval%2F02.1_agent_evaluation&version=1">


In [0]:
%pip install -U -qqqq mlflow>=3.1.4 langchain==0.3.27 langgraph==0.6.11 databricks-langchain pydantic databricks-agents unitycatalog-langchain[databricks] databricks-feature-engineering==0.12.1 protobuf<5  cryptography<43 databricks-mcp
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%run ../_resources/01-setup

USE CATALOG `main_build`
using catalog.database `main_build`.`dbdemos_ai_agent`


data already exists


## 1/ Build and register our agent

### 1.1/ Define our agent configuration
Let's first update our configuration file with the tools we want our langchain agent to use, a basic system prompt and the endpoint we want to use.

In [0]:
import yaml
import mlflow

rag_chain_config = {
    "config_version_name": "first_config",
    "input_example": [{"role": "user", "content": "Give me the orders for john21@example.net"}],
    "uc_tool_names": [f"{catalog}.{dbName}.*"],
    "system_prompt": "Your job is to provide customer help. call the tool to answer.",
    "llm_endpoint_name": LLM_ENDPOINT_NAME,
    "max_history_messages": 20,
    "retriever_config": None,
    "mcp_server_urls": [] 
}
try:
    with open('agent_config.yaml', 'w') as f:
        yaml.dump(rag_chain_config, f)
except:
    print('pass to work on build job')
model_config = mlflow.models.ModelConfig(development_config='agent_config.yaml')

pass to work on build job


We created our AGENT using langchain in the `agent.py` file. You can explore it to see the code behind the scene.

In this notebook, we'll keep it simple and just import it and send a request to explore its internal tracing with MLFlow Trace UI:

In [0]:
from agent import AGENT

# Correct request format
request_example = "Give me the information about john21@example.net"
answer = AGENT.predict({"input":[{"role": "user", "content": request_example}]})

2025/11/12 21:58:26 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 21:58:26 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.


Trace(trace_id=tr-4b27282365bfa70a56d21d17369d4ca0)


### 1.2/ Open MLFlow tracing

<img src="https://i.imgur.com/tNYUHdC.gif" style="float: right" width="700px">

Open now the experiment from the right notebook menu. You'll see in the traces the message we just sent: `Give me the information about john21@example.net`.

MLFlow keeps track of all the input/output and internal tracing so that we can analyze existing request, and create better evaluation dataset over time!

Not only MLFlow traces all your agent request, but you can also easily capture end-users feedback to quickly detect which answer was wrong and improve your agent accordingly! 

*We'll show you how to capture feedback when we'll deploy the application!*

### 1.3/ Log the `agent` as an MLflow model

This looks good! Let's log the agent in our MLFlow registry using the [agent]($./agent) python file to avoid any serialization issue. See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

*Note that we'll also pass the list of Databricks resources (functions, warehouse etc) that our agent need to use to properly work. This will handle the permissions for us during its deployment!*

In [0]:
import mlflow
def log_customer_support_agent_model(resources, request_example):
    with mlflow.start_run(run_name=model_config.get('config_version_name')):
        return mlflow.pyfunc.log_model(
            name="agent",
            python_model="agent.py",
            model_config="agent_config.yaml",
            input_example={"input": [{"role": "user", "content": request_example}]},
            resources=resources, # Determine Databricks resources (endpoints, fonctions, vs...) to specify for automatic auth passthrough at deployment time
            extra_pip_requirements=["databricks-connect"]
        )
logged_agent_info = log_customer_support_agent_model(AGENT.get_resources(), request_example)

🔗 View Logged Model at: https://xxxx.cloud.databricks.com/ml/experiments/6b46e63d8b9947a09920c46ad4ea44a3/models/m-7bdd0a85ff2a45a2a03cdd2fb8370f3f?o=1660015457675682
2025/11/12 21:58:41 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 21:58:41 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.
2025/11/12 21:58:41 INFO mlflow.pyfunc: Predicting on input example to validate output
2025/11/12 21:58:41 WARNING mlflow.tracing.fluent: No active trace found. Please crea

### 1.4/ Let's load and try our model
Our model is saved on MLFlow! Let's load it and give it a try. We'll wrap our predict function so that we can extract more easily the final answer, and also make our evaluation easier:

In [0]:
import pandas as pd
# Load the model and create a prediction function
loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")
def predict_wrapper(question):
    # Format for chat-style models
    model_input = pd.DataFrame({
        "input": [[{"role": "user", "content": question}]]
    })
    response = loaded_model.predict(model_input)
    return response['output'][-1]['content'][-1]['text']

answer = predict_wrapper("Give me the orders for john21@example.net.")
print(answer)

2025/11/12 21:59:17 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 21:59:17 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.


Here are your current orders:

1. Premium ADSL Plan (ADSL service)
   - Monthly charge: $47
   - Started: December 20, 2022
   - 12-month contract
   - Status: Active
   - Autopay: Not enabled

2. Premium Mobile Plan (Mobile service)
   - Monthly charge: $84
   - Started: June 19, 2023
   - No contract term (month-to-month)
   - Status: Active
   - Autopay: Enabled

Is there anything specific about these orders you'd like to know more about?


Trace(trace_id=tr-575304d08da5e266e5c4bc48126f031c)

## 2/ Evaluation

### 2.1/ Evaluate the agent with [Agent Evaluation](https://docs.databricks.com/aws/en/mlflow3/genai/eval-monitor/)

We prepared an evaluation dataset ready to use as part of this demo. However, multiple strateges exist:

- create your own evaluation dataset (what we'll do)
- use existing traces from MLFlow and add them to your dataset (from the API or your experiment UI)
- use Databricks genai eval synthetic dataset creation (see the [PDF RAG notebook]($../03-knowledge-base-rag/03.1-pdf-rag-tool) for an example)
- Create labeling session where you can get insights from expert, using the MLFlow UI directly!

Note: you can also select the existing LLM call from the traces and add to your eval dataset with the UI, or use the API directly:

```
traces = mlflow.search_traces(filter_string=f"attributes.timestamp_ms > {ten_minutes_ago} AND attributes.status = 'OK'", order_by=["attributes.timestamp_ms DESC"])
```

In [0]:
%run ../_resources/04-eval-dataset-generation

DataFrame[]

inputs,predictions,expectations
List(What is the phone number of john21@example.net?),,"List(List(The phone number of john21@example.net is 001-581-896-0013x3890., The phone number associated with john21@example.net belongs to Danielle Johnson.))"
List(List all orders placed by blairamanda@example.com.),,"List(List(There is no information about orders placed by blairamanda@example.com in the provided customer record., The customer record only contains personal information about Melanie Munoz with email blairamanda@example.com., The record shows customer details but does not include order history., No order data is available in the provided information.))"
List(What is the current subscription status for onelson@example.net?),,"List(List(The current subscription status for onelson@example.net is Active., The customer associated with onelson@example.net has an Active subscription status., Jessica Martin's subscription status is Active.))"
List(Show billing details for laurahenderson@example.org.),,"List(List(The email address is laurahenderson@example.org., The customer's first name is Brandon., The customer's last name is Rodriguez., The phone number is (669)878-4801x8451., The address is 0482 Monica Hills, East Nathaniel, GA, 71198., The customer has a Bronze membership level., The customer has spent $3.23., The customer has made 37 purchases., The customer has 67 loyalty points., The customer has an Active status., The customer joined on 2022-05-14., The customer has a Family account type.))"
List(Does tracy15@example.com have any unpaid invoices?),,"List(List(The customer record does not contain information about unpaid invoices., The record shows that Sandra Sellers has the email address tracy15@example.com., The customer record includes contact information and membership details but no invoice or payment status information., Based on the provided information, it's not possible to determine if tracy15@example.com has any unpaid invoices.))"
List(Which products did briannasmith@example.net purchase?),,"List(List(The customer record does not contain information about products purchased by briannasmith@example.net., The customer record only contains personal information, contact details, and account status for Matthew Chapman., No purchase history or product information is included in the provided customer record.))"
List(What is the loyalty tier of lauren13@example.org?),,"List(List(The loyalty tier of lauren13@example.org is Bronze., The customer associated with lauren13@example.org has a Bronze loyalty tier., Lauren13@example.org belongs to the Bronze loyalty tier.))"
List(When did smitchell@example.net register as a customer?),,"List(List(The customer registered on June 4, 2013., The registration date is 2013-06-04., smitchell@example.net has been a customer since June 4, 2013., The customer record shows a registration date of 2013-06-04.))"
List(Summarize all subscriptions held by harveyrobert@example.net.),,"List(List(The customer with email harveyrobert@example.net has a Bronze subscription., The subscription is currently Active., The subscription was started on February 11, 2021., The subscription is in the Family category.))"
List(What city does contrerasangela@example.net live in?),,"List(List(Cheryl Archer lives in North Cynthiaview., The customer with email contrerasangela@example.net lives in North Cynthiaview., North Cynthiaview is the city where Cheryl Archer resides., The city of residence for the customer with email contrerasangela@example.net is North Cynthiaview.))"


In [0]:
eval_example = spark.read.json(f"/Volumes/{catalog}/{dbName}/{volume_name}/eval_dataset")
display(eval_example)

expectations,inputs,predictions
"List(List(The current subscription status for onelson@example.net is Active., The customer with email onelson@example.net has an Active subscription status., Jessica Martin's subscription status is Active.))",List(What is the current subscription status for onelson@example.net?),
"List(List(The email address is laurahenderson@example.org., The customer's first name is Brandon., The customer's last name is Rodriguez., The phone number is (669)878-4801x8451., The address is 0482 Monica Hills, East Nathaniel, GA, 71198., The customer has a Bronze membership level., The customer has spent $3.23., The account is Active., The account was created on 2022-05-14., The customer has made 37 purchases., The customer has 67 loyalty points.))",List(Show billing details for laurahenderson@example.org.),
"List(List(The customer record does not contain information about unpaid invoices., The record shows that Sandra Sellers has an email address of tracy15@example.com., The record shows the customer has an Active status., The record shows the customer has a Silver membership level., The record does not include any invoice or payment history information.))",List(Does tracy15@example.com have any unpaid invoices?),
"List(List(The customer registered on June 4, 2013., The registration date is 2013-06-04., smitchell@example.net has been a customer since June 4, 2013., The customer record shows a registration date of 2013-06-04.))",List(When did smitchell@example.net register as a customer?),
"List(List(The customer with email harveyrobert@example.net is Shelly Hudson., Shelly Hudson has a Bronze subscription., The subscription is currently Active., The subscription started on February 11, 2021., The subscription is for the Family plan., The customer has a rating of 4.48., The customer has made 1 purchase., The customer has accumulated 82 points.))",List(Summarize all subscriptions held by harveyrobert@example.net.),
"List(List(Cheryl Archer lives in North Cynthiaview., The customer with email contrerasangela@example.net lives in North Cynthiaview., North Cynthiaview is the city where Cheryl Archer resides., The city of residence for the customer with email contrerasangela@example.net is North Cynthiaview.))",List(What city does contrerasangela@example.net live in?),
"List(List(The churn risk score for carlsonmichael@example.com is 2.11., The customer with email carlsonmichael@example.com has a churn risk score of 2.11.))",List(What is the churn risk score for carlsonmichael@example.com?),
"List(List(The customer value score for lbyrd@example.net is 2.27., David Fisher has a customer value score of 2.27., The customer with email lbyrd@example.net has a value score of 2.27.))",List(What is the customer value score for lbyrd@example.net?),
"List(List(The customer record does not contain information about whether autopay is enabled., The data provided does not include any autopay status information., There is no field in the given customer record that indicates autopay status., Based on the provided information, it cannot be determined if autopay is enabled for gibsonolivia@example.net's account.))",List(Is autopay enabled for gibsonolivia@example.net's account?),
"List(List(The customer with email aramirez@example.com is a Business customer., The customer type for aramirez@example.com is Business.))","List(What type of customer is aramirez@example.com (e.g., Individual, Business)?)",


### 2.2/ Create our MLFlow dataset
Let's use the API to create our dataset. You can also directly do it from the Experiment UI!

In [0]:
import mlflow
import mlflow.genai.datasets

eval_dataset_table_name = f"{catalog}.{dbName}.ai_agent_mlflow_eval"

try:
  eval_dataset = mlflow.genai.datasets.get_dataset(eval_dataset_table_name)
except Exception as e:
  if 'does not exist' in str(e):
    eval_dataset = mlflow.genai.datasets.create_dataset(eval_dataset_table_name)
    # Add your examples to the evaluation dataset
    eval_dataset.merge_records(eval_example)
    print("Added records to the evaluation dataset.")

# Preview the dataset
display(eval_dataset.to_df())

dataset_record_id inputs expectations source tags create_time last_update_time created_by last_updated_by 017581eb-d496-45a7-b7e2-82137c1f3b96 List(What is the churn risk score for carlsonmichael@example.com?) List(List(The churn risk score for carlsonmichael@example.com is 2.11., The customer with email carlsonmichael@example.com has a churn risk score of 2.11.), null) null null 2025-08-13T16:00:16.347Z 2025-08-13T16:00:16.347Z quentin.ambard@databricks.com quentin.ambard@databricks.com 02584276-ded2-4873-af0a-53c731bf46fe List(How do I troubleshoot Error Code 1001: Invalid Return Authorization when my return request submission fails?) List(List(Verify the return authorization number against system records., Initiate a new return authorization if the existing one is missing or invalid., Ensure the return authorization is active and within a valid 14-day period., Update the return request with correct authorization details., Contact technical support if the error persists.), List(List(**Error Code 1001: Invalid Return Authorization**

**Description:** The return request was submitted without proper authorization or documentation.

**Symptoms:** Customer reports inability to proceed with return; system displays error 1001 during submission.

**Root Causes:** Missing or incorrect return authorization number, expired authorization, or system misconfiguration.

## Resolution Steps:

1. Verify the return authorization number provided by the customer against the system records.
2. If missing or invalid, initiate a new return authorization following the standard procedure.
3. Ensure the authorization is active and within the valid period (typically 14 days from issue).
4. Update the customer's return request with the correct authorization details.
5. If system error persists, contact technical support for system diagnostics., dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/return_policy_2024_manual.pdf))) List(List(**Error Code 1001: Invalid Return Authorization**

**Description:** The return request was submitted without proper authorization or documentation.

**Symptoms:** Customer reports inability to proceed with return; system displays error 1001 during submission.

**Root Causes:** Missing or incorrect return authorization number, expired authorization, or system misconfiguration.

## Resolution Steps:

1. Verify the return authorization number provided by the customer against the system records.
2. If missing or invalid, initiate a new return authorization following the standard procedure.
3. Ensure the authorization is active and within the valid period (typically 14 days from issue).
4. Update the customer's return request with the correct authorization details.
5. If system error persists, contact technical support for system diagnostics., dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/return_policy_2024_manual.pdf), null, null) null 2025-09-24T13:48:56.652Z 2025-09-24T13:48:56.652Z quentin.ambard@databricks.com quentin.ambard@databricks.com 043dd657-e3d3-4b91-a0a8-3db26b658d4a List(What steps should I take if my device will not power on?) List(List(Check the power connection and outlet functionality., Verify that the power adapter is securely connected., Test with a known working power adapter if available., Inspect the device for physical damage., Replace the device if a hardware failure is confirmed.), List(List(### Scenario 1: Device not powering on

1. Check power connection and outlet functionality.
2. Verify power adapter is securely connected.
3. Test with a known working power adapter if available.
4. Inspect device for physical damage.
5. Replace device if hardware failure is confirmed., dbfs:/Volumes/main/dbdemos_ai_agent/raw_data/pdf_documentation/smart_home_net_manual.pdf))) List(List(### Scenario 1: Device not powering on

1. Check power connection and outlet functionality.
2. Verify power adapter is securely connected.
3. Test with a known working power adapter if available.
4. 

## 2.3/ Adding guidelines to track our agent behavior

<img src="https://i.imgur.com/M3kLBHF.gif" style="float:right" width="700px">

MLFlow 3.0 lets you create custom guidelines to evaluate your agent behavior.

We'll use a few of the built-in one, and add a custome `Guidelines` on steps and reasoning: we want our LLM to output the answer without mentioning the internal tools it has.

In [0]:
from mlflow.genai.scorers import RetrievalGroundedness, RelevanceToQuery, Safety, Guidelines

def get_scorers():
    return [
        RetrievalGroundedness(),  # Checks if email content is grounded in retrieved data
        RelevanceToQuery(),  # Checks if email addresses the user's request
        Safety(),  # Checks for harmful or inappropriate content
        Guidelines(
            guidelines="""
            Reponse must be done without showing reasoning.
            - don't mention that you need to look up things
            - do not mention tools or function used
            - do not tell your intermediate steps or reasoning
            """,
            name="steps_and_reasoning",
        )
    ]

scorers = get_scorers()

## 2.4/ Run the evaluations against our guidelines

That's it, let's now evaluate our dataset with our guidelines:

In [0]:
with mlflow.start_run(run_name='eval_with_no_reasoning_instructions'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=predict_wrapper, scorers=scorers)

2025/11/12 22:00:02 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2025/11/12 22:00:02 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
2025/11/12 22:00:02 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.
2025/11/12 22:00:07 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.


Evaluating:   0%|          | 0/114 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

[Trace(trace_id=tr-09a6e8b2a78fffb92c3482364594963e), Trace(trace_id=tr-b2c20398187d6d851731e01f857c197e), Trace(trace_id=tr-9a2797cd0534fe39a31dd7c21df11946), Trace(trace_id=tr-139e4148803e968131e1b3261467663b), Trace(trace_id=tr-0f9fd469ab35069f544c0a61d71e86fb), Trace(trace_id=tr-828c2e20388224ed6900707ced377d32), Trace(trace_id=tr-5fa5008cfd7db29f653411b9d6d80b9d), Trace(trace_id=tr-083b84d8e9d300062a853b51cdd7f2bf), Trace(trace_id=tr-fcbfd0caa2b4a446d5e57708093e44e4), Trace(trace_id=tr-c6d40575dbdf955181ac1c2f320e29dc)]

## 3/ Improving our eval metrics with a better system prompt

As we can see in the eval, the agent emits a lot of information on the internal tools and steps. 
For example; it would mention things like:

`First, I need to find his customer record using his email address. Since I don't have Thomas Green's email address yet, I need to ask for it.`

While this is good reasoning, we do not want this in the final answer!

### 3.1/ Deploying a new model version with a better system prompt

Let's update our system prompt with better instruction to avoid this behavior, and run our eval to make sure this improved!

In [0]:
try:
    config = yaml.safe_load(open("agent_config.yaml"))
    config["config_version_name"] = "better_prompt"
    config["system_prompt"] = (
        "You are a telco assistant. Call the appropriate tool to help the user with billing, support, or account info. "
        "DO NOT mention any internal tool or reasoning steps in your final answer. Do not say according to records or imply that you are looking up information."
    )
    yaml.dump(config, open("agent_config.yaml", "w"))
except Exception as e:
    print(f"Skipped update - ignore for job run - {e}")

Skipped update - ignore for job run - [Errno 22] Invalid argument


In [0]:
# Let's relog our agent in MLflow to capture the new prompt
logged_agent_info = log_customer_support_agent_model(AGENT.get_resources(), request_example)

🔗 View Logged Model at: https://xxxx.cloud.databricks.com/ml/experiments/6b46e63d8b9947a09920c46ad4ea44a3/models/m-58f4c6ba2f8e48dd9707f354a3bdeeef?o=1660015457675682
2025/11/12 22:02:37 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 22:02:37 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.
2025/11/12 22:02:38 INFO mlflow.pyfunc: Predicting on input example to validate output
2025/11/12 22:02:38 WARNING mlflow.tracing.fluent: No active trace found. Please crea

In [0]:
# Load the model to be used in evaluation via `predict_wrapper`
loaded_model = mlflow.pyfunc.load_model(f"runs:/{logged_agent_info.run_id}/agent")

with mlflow.start_run(run_name='eval_with_reasoning_instructions'):
    results = mlflow.genai.evaluate(data=eval_dataset, predict_fn=predict_wrapper, scorers=scorers)

2025/11/12 22:03:10 WARNING mlflow.pyfunc: You have manually traced predict with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
2025/11/12 22:03:10 WARNING mlflow.pyfunc: You have manually traced predict_stream with @mlflow.trace, but this is unnecessary with ResponsesAgent subclasses. You can remove the @mlflow.trace decorator and it will be automatically traced.
Function name main_build__dbdemos_ai_agent__get_customer_billing_and_subscriptions is too long, truncating to 64 characters _build__dbdemos_ai_agent__get_customer_billing_and_subscriptions.
2025/11/12 22:03:12 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset.
2025/11/12 22:03:12 WARNING mlflow.tracing.fluent: No active trace found. Please create a span using `mlflow.start_span` or `@mlflow.trace` before calling `mlflow.update_current_trace`.
2025/11/12 22:03:17 WARNING m

Evaluating:   0%|          | 0/114 [Elapsed: 00:00, Remaining: ?] 

<!DOCTYPE html>
 
 
 Evaluation output 
 
 
 
 
 
 
 
 
 View evaluation results.

[Trace(trace_id=tr-47b590dcdf9ec50b8d6e189a8ee46577), Trace(trace_id=tr-e7a83f8334820dfd19ebcdf289d8549b), Trace(trace_id=tr-384cd3c6dcc1af9b164f73f1091e8872), Trace(trace_id=tr-e90e0f259000d39426231aaa26ee9bf4), Trace(trace_id=tr-d9d87e5df14695227fffe95e99c7b368), Trace(trace_id=tr-2518525e766b209331dc6e6922fb0731), Trace(trace_id=tr-39de983892b48efcb9d265b79409522f), Trace(trace_id=tr-70c090b61be304e593dd380b1130a4cd), Trace(trace_id=tr-7056e6b178e4cc76dc1def641ac4b253), Trace(trace_id=tr-ffe4fcb0cf088f43ebcd883acd75b03f)]

Open your experiment and check the results!

Select the previous run and this one, and compare them. You should see some improvements!

## 4/ Deploy our agent as an endpoint!

Everything looks good! Our latest version now has decent eval score. Let's deploy it as a realtime endpoint for our end user chat application.

### 4.1/ Register our new model version to Unity Catalog


In [0]:
from mlflow import MlflowClient
UC_MODEL_NAME = f"{catalog}.{dbName}.{MODEL_NAME}"

# register the model to UC
client = MlflowClient()
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=UC_MODEL_NAME, tags={"model": "customer_support_agent"})
client.set_registered_model_alias(name=UC_MODEL_NAME, alias="model-to-deploy", version=uc_registered_model_info.version)

# Create HTML link to created agent
displayHTML(f'<a href="/explore/data/models/{catalog}/{dbName}/{MODEL_NAME}" target="_blank">Open Unity Catalog to see Registered Agent</a>')

Registered model 'main.dbdemos_ai_agent.dbdemos_ai_agent_demo' already exists. Creating a new version of this model...
🔗 Created version '22' of model 'main.dbdemos_ai_agent.dbdemos_ai_agent_demo': https://xxxx.cloud.databricks.com/explore/data/models/main_build/dbdemos_ai_agent/dbdemos_ai_agent_demo/version/22?o=1660015457675682


Open Unity Catalog to see Registered Agent

### 4.2/ Deploy the agent

Let's now start our model endpoint:

In [0]:
from databricks import agents
# Deploy the model to the review app and a model serving endpoint
if len(agents.get_deployments(model_name=UC_MODEL_NAME, model_version=uc_registered_model_info.version)) == 0:
  agents.deploy(UC_MODEL_NAME, uc_registered_model_info.version, endpoint_name=ENDPOINT_NAME, tags = {"project": "dbdemos"})


    Deployment of main.dbdemos_ai_agent.dbdemos_ai_agent_demo version 22 initiated.  This can take up to 15 minutes and the Review App & Query Endpoint will not work until this deployment finishes.

    View status: https://xxxx.cloud.databricks.com/ml/endpoints/dbdemos_ai_agent_demo_main_build_dbdemos_ai_agent
    Review App: https://xxxx.cloud.databricks.com/ml/review-v2/chat?endpoint=dbdemos_ai_agent_demo_main_build_dbdemos_ai_agent

You can refer back to the links above from the endpoint detail page at https://xxxx.cloud.databricks.com/ml/endpoints/dbdemos_ai_agent_demo_main_build_dbdemos_ai_agent.


## Next: adding a tool to answer questions about our knowledge base (RAG + Vector Search on PDF)

Our model is working well, but it can't answer specific questions that our customer support might have about their subscription.

For example, if we ask our Agent how to solve a specific error code in our WIFI router, it'll fail as it doesn't have any valuable information about it.

Open the [03-knowledge-base-rag/03.1-pdf-rag-tool]($../03-knowledge-base-rag/03.1-pdf-rag-tool)